# 01 · Data check: MD-SynReal12K

Run this once after downloading the dataset. It answers:
1. How many images are in each split (Train / Validation / Test) and each class (Real / AI)?
2. Which file formats does each class use? (If real = JPEG and AI = PNG, we must convert all images to one format.)
3. What are the image sizes?
4. Are there exact duplicate files, including across splits?

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change this to where you put the downloaded zip in your Drive
ZIP_PATH = '/content/drive/MyDrive/MD-SynReal12k/dataset.zip'
DATA_DIR = '/content/data'

import os
if not os.path.exists(DATA_DIR):
    !unzip -q "$ZIP_PATH" -d "$DATA_DIR"
print('Unzipped to', DATA_DIR)

In [ ]:
import os, hashlib
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd
from PIL import Image

IMG_EXT = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

import re

def guess(parts, options):
    # Match whole words only, so 'train' is not read as 'ai'
    for p in parts:
        tokens = [t for t in re.split(r'[^a-z]+', p.lower()) if t]
        for key, words in options.items():
            if any(t == w or t.startswith(w) for t in tokens for w in words):
                return key
    return 'unknown'

SPLITS = {'train': ['train'], 'val': ['val'], 'test': ['test']}
CLASSES = {'real': ['real', 'authentic'], 'ai': ['fake', 'synthetic', 'ai', 'generated']}

rows = []
for path in Path(DATA_DIR).rglob('*'):
    if path.suffix.lower() in IMG_EXT:
        parts = path.relative_to(DATA_DIR).parts[:-1]
        rows.append({
            'path': str(path),
            'split': guess(parts, SPLITS),
            'label': guess(parts, CLASSES),
            'ext': path.suffix.lower(),
            'folder': '/'.join(parts),
        })

df = pd.DataFrame(rows)
print('Total images:', len(df))

## 1. Images per split and class

In [ ]:
print(pd.crosstab(df['split'], df['label'], margins=True))
print()
print('Folders found:')
print(df['folder'].value_counts().to_string())

## 2. File formats per class

**Important:** if one class is all `.jpg` and the other all `.png`, a model can cheat by learning the file format. In that case we convert every image to the same format in notebook 02.

In [ ]:
print(pd.crosstab(df['label'], df['ext'], margins=True))

## 3. Image sizes (sample of 300 per class)

In [ ]:
sizes = []
for label, group in df.groupby('label'):
    for p in group.sample(min(300, len(group)), random_state=0)['path']:
        with Image.open(p) as im:
            sizes.append({'label': label, 'size': f'{im.width}x{im.height}', 'mode': im.mode})
sizes = pd.DataFrame(sizes)
print(pd.crosstab(sizes['size'], sizes['label']))
print()
print(pd.crosstab(sizes['mode'], sizes['label']))

## 4. Exact duplicate files

In [ ]:
def md5(p, chunk=1 << 20):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

df['md5'] = df['path'].map(md5)
dups = df[df.duplicated('md5', keep=False)].sort_values('md5')
print('Files that have an exact copy:', len(dups))
cross = dups.groupby('md5')['split'].nunique()
print('Duplicate groups that appear in more than one split (data leakage):', int((cross > 1).sum()))

## 5. Look at a few images

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for row, label in enumerate(['real', 'ai']):
    sample = df[df['label'] == label].sample(6, random_state=1)
    for ax, (_, r) in zip(axes[row], sample.iterrows()):
        ax.imshow(Image.open(r['path']).convert('RGB'))
        ax.set_title(f"{label} · {r['ext']}")
        ax.axis('off')
plt.tight_layout()
os.makedirs('results/figures', exist_ok=True)
plt.savefig('results/figures/sample_images.png', dpi=120)
plt.show()

## Summary to share with the team

Write down:
- Total images and per split/class counts
- Whether the formats differ between classes (yes/no)
- Image sizes
- Number of duplicates and cross-split duplicates